In [1]:
import numpy as np
import healpy as hp
import matplotlib.pylab as plt
%matplotlib inline
import rubin_sim.splat as splat
import copy 

import sqlite3
import pandas as pd
from os.path import basename

import glob

In [2]:
# Define a report element that we want to run. Needs branch maf-proto of rubin_sim to run
class AlertsperyearElement(splat.BaseReportElement):
    """Estimate how many alerts in certain years one would get
    """
    def compute_elements(self,visits_df=None, visits_array=None, nside=32, bands="ugrizy",
                         seeing_limit=1.2, n_for_template=6, years=np.arange(10)+1):
        
        times = np.arange(3653)
        metric = splat.AccumulateCountMetric(times)
        
        is_too = np.array(["ToO" in note for note in visits_array["scheduler_note"]])
        is_ddf = np.array(["DD" in note for note in visits_array["scheduler_note"]])

        count_arrays = {}
        good_seeing_count = {}
        count_subbed = {}
        alerts_years = np.zeros(years.size)
        
        for band in bands:
            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) & (visits_array["band"] == band))
            sub_data = visits_array[indx]
            sl = splat.Slicer(nside=nside, missing=0)
            count_arrays[band] = sl(sub_data, metric)

            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) &
                            (visits_array["band"] == band) &
                            (visits_array["seeingFwhm500"] < seeing_limit))
            sub_data = visits_array[indx]
            sl = splat.Slicer(nside=nside, missing=0)
            good_seeing_count[band] = sl(sub_data, metric)
        
            # Need to find the spot where good_seeing_count >= n_for_template
            mask = np.ones(good_seeing_count[band].shape) 
            mask[np.where(good_seeing_count[band] >= 6)] = 0
            count_masked = count_arrays[band] * mask
            count_subbed[band] = count_arrays[band] - np.max(count_masked, axis=1)[:,np.newaxis]
            count_subbed[band][np.where(count_subbed[band] < 0)] = 0

            for i,year in enumerate(years):
                alerts = count_subbed[band][:, int(365.25 * (year))] - count_subbed[band][:, int(365.25 * (year-1))]
                alerts_years[i] += np.sum(alerts)
        
        return alerts_years, count_subbed, None


In [3]:
db_files = ['baseline_v5.3.7_10yrs.db', 
            'template_var_1_v5.3.7_10yrs.db',
            'template_var_2_v5.3.7_10yrs.db',
            'template_var_3_v5.3.7_10yrs.db',
            'three_stripes_v5.3.7_10yrs.db',
            'three_stripes_kr_v5.3.7_10yrs.db',
            'desi_tskr_v5.3.7_10yrs.db',
            'desi_stripes_delayed_v5.3.7_10yrs.db',
            'desi_stripes_v5.3.7_10yrs.db',
            "double_stripes_timing_bands_v5.3.7_10yrs.db",
            "three_stripes_or_v5.3.7_10yrs.db",
            "delay_roll_v5.3.7_10yrs.db"
            ]

In [4]:
alerts = {}
counts_subtracted = {}

for filename in db_files:
    element = AlertsperyearElement(observations=filename)
    alerts_years, counts_subbed, count_arrays, = element()
    alerts[filename.replace(".db", "")] = alerts_years
    counts_subtracted[filename.replace(".db", "")] = counts_subbed
    


In [5]:
# potential alerts / nside summed over bands?

In [10]:
table = pd.DataFrame.from_dict(alerts).T
table.columns = np.arange(10)+1

In [13]:
print("Number of possible HEALpix visits with alerts in year N")
table

Number of possible HEALpix visits with alerts in year N


,1,2,3,4,5,6,7,8,9,10
baseline_v5.3.7_10yrs,149223.0,397881.0,325479.0,396311.0,426711.0,449641.0,515963.0,446579.0,524363.0,419569.0
template_var_1_v5.3.7_10yrs,142439.0,378454.0,327560.0,398138.0,426121.0,449408.0,515459.0,448899.0,525780.0,419590.0
template_var_2_v5.3.7_10yrs,139374.0,390814.0,327999.0,397488.0,426449.0,450441.0,516763.0,448465.0,523878.0,419246.0
template_var_3_v5.3.7_10yrs,149415.0,384979.0,327135.0,398838.0,426580.0,449339.0,515683.0,447983.0,525219.0,419354.0
three_stripes_v5.3.7_10yrs,149226.0,400100.0,323318.0,393990.0,428177.0,447132.0,513309.0,451424.0,521283.0,414941.0
three_stripes_kr_v5.3.7_10yrs,185802.0,379917.0,315488.0,388204.0,418569.0,443359.0,512766.0,447321.0,525380.0,419802.0
desi_tskr_v5.3.7_10yrs,185797.0,375212.0,313757.0,386026.0,417924.0,445387.0,513100.0,446532.0,526353.0,418922.0
desi_stripes_delayed_v5.3.7_10yrs,149249.0,394885.0,322929.0,394570.0,424526.0,449373.0,514817.0,448525.0,525908.0,419948.0
desi_stripes_v5.3.7_10yrs,176236.0,370892.0,315254.0,392440.0,424194.0,448067.0,515302.0,448560.0,524620.0,418459.0
double_stripes_timing_bands_v5.3.7_10yrs,182943.0,376282.0,313140.0,386523.0,418060.0,443193.0,518045.0,448421.0,520145.0,417863.0
